# Telco Churn — merge the five IBM tables into a modelling dataset

Reads the five source tables in `data/` (downloaded from IBM's public server) and
writes analysis-ready CSVs.

**Two files out, not one — deliberately.**

| File | Contents | Use |
|---|---|---|
| `data/telco_churn_model.csv` | features + `Churn Value` | the only file modelling code should read |
| `data/telco_churn_reference.csv` | `Customer ID` + every leaky/outcome field | post-hoc validation only — **never** a feature source |

Splitting them is the point. `Churn Reason` was genuinely useful earlier for checking
whether SHAP's drivers matched what churners actually said — but it only exists for
churners, so it can never be a feature. Keeping it in a separate file means it stays
available for analysis while being impossible to merge into training by accident.

**What gets dropped, and why**

1. *Leakage* — fields recorded with or after the outcome. `Satisfaction Score` scores
   AUC 0.95 on its own and `Churn Score` 0.94, versus 0.86 for a fully tuned model on
   real features. That is the outcome wearing a different name; you will not have next
   quarter's satisfaction survey when scoring next quarter's customers.
2. *Exact duplicates* — several flags are bit-for-bit functions of another column
   (`Under 30` is `Age < 30`). Keeping both splits a feature's importance across two
   rows and makes attribution lie, without adding a thing. The notebook proves each
   one before dropping it rather than trusting this list.
3. *Constants and IDs* — no variance, or an identifier.

Feature *engineering* (tenure buckets, spend ratios) deliberately stays in the
modelling notebook. This file's job is a clean, honest column set.

In [2]:
import numpy as np
import pandas as pd
from pathlib import Path
from sklearn.metrics import roc_auc_score

DATA = Path("data")
OUT_MODEL = DATA / "telco_churn_model.csv"
OUT_REF = DATA / "telco_churn_reference.csv"

KEY = "Customer ID"
TARGET = "Churn Value"

dem = pd.read_excel(DATA / "Telco_customer_churn_demographics.xlsx")
loc = pd.read_excel(DATA / "Telco_customer_churn_location.xlsx")
pop = pd.read_excel(DATA / "Telco_customer_churn_population.xlsx")
svc = pd.read_excel(DATA / "Telco_customer_churn_services.xlsx")
sta = pd.read_excel(DATA / "Telco_customer_churn_status.xlsx")

for n, d in [("demographics", dem), ("location", loc), ("population", pop),
             ("services", svc), ("status", sta)]:
    print(f"{n:13s} {d.shape[0]:>5} rows x {d.shape[1]:>2} cols")

demographics   7043 rows x  9 cols
location       7043 rows x 10 cols
population     1671 rows x  3 cols
services       7043 rows x 31 cols
status         7043 rows x 12 cols


## 1. Audit — prove the duplicates and the leaks

Nothing below is dropped on faith. Every duplicate is verified bit-for-bit against the
data, and every leak is measured. If IBM changes the tables, these checks fail loudly
instead of silently keeping a bad column.

In [3]:
def is_duplicate(label, a, b):
    same = bool((a == b).all())
    print(f"  {'DUPLICATE ' if same else 'distinct  '} {label}")
    return same

print("=== exact-duplicate checks ===")
dup = {}
dup["Under 30"]          = is_duplicate("Under 30          == (Age < 30)",
                                        dem["Under 30"] == "Yes", dem["Age"] < 30)
dup["Senior Citizen"]    = is_duplicate("Senior Citizen    == (Age >= 65)",
                                        dem["Senior Citizen"] == "Yes", dem["Age"] >= 65)
dup["Dependents"]        = is_duplicate("Dependents        == (Number of Dependents > 0)",
                                        dem["Dependents"] == "Yes", dem["Number of Dependents"] > 0)
dup["Referred a Friend"] = is_duplicate("Referred a Friend == (Number of Referrals > 0)",
                                        svc["Referred a Friend"] == "Yes", svc["Number of Referrals"] > 0)
dup["Internet Service"]  = is_duplicate("Internet Service  == (Internet Type not null)",
                                        svc["Internet Service"] == "Yes", svc["Internet Type"].notna())

# Total Revenue is arithmetic, not a flag — check it as such.
_calc = (svc["Total Charges"] + svc["Total Extra Data Charges"]
         + svc["Total Long Distance Charges"] - svc["Total Refunds"])
_d = (svc["Total Revenue"] - _calc).abs().max()
dup["Total Revenue"] = _d < 0.01
print(f"  {'DUPLICATE ' if dup['Total Revenue'] else 'distinct  '} "
      f"Total Revenue     == Charges + Extra + LongDist - Refunds  (max diff {_d:.4f})")

print("\n  Married kept: verified distinct from Dependents ->",
      not bool(((dem["Married"] == "Yes") == (dem["Dependents"] == "Yes")).all()))

=== exact-duplicate checks ===
  DUPLICATE  Under 30          == (Age < 30)
  DUPLICATE  Senior Citizen    == (Age >= 65)
  DUPLICATE  Dependents        == (Number of Dependents > 0)
  DUPLICATE  Referred a Friend == (Number of Referrals > 0)
  DUPLICATE  Internet Service  == (Internet Type not null)
  DUPLICATE  Total Revenue     == Charges + Extra + LongDist - Refunds  (max diff 0.0000)

  Married kept: verified distinct from Dependents -> True


In [4]:
print("=== leakage severity: AUC of each status field ALONE ===")
print("   (a fully tuned model on real features reaches ~0.86)\n")
for c in ["Satisfaction Score", "Churn Score", "CLTV"]:
    a = roc_auc_score(sta[TARGET], sta[c])
    a = max(a, 1 - a)                      # sign-agnostic
    flag = "LEAK" if a > 0.90 else "suspect"
    print(f"  {flag:8s} {c:20s} AUC {a:.4f}")

print(f"\n  LEAK     Customer Status      values {sta['Customer Status'].unique().tolist()}"
      f"  <- restates the target")
print(f"  LEAK     Churn Label          identical to {TARGET}: "
      f"{bool(((sta['Churn Label'] == 'Yes') == (sta[TARGET] == 1)).all())}")
print(f"  LEAK     Churn Category       null for {int(sta['Churn Category'].isna().sum())}"
      f" of {len(sta)} (populated only for churners)")
print(f"  LEAK     Churn Reason         null for {int(sta['Churn Reason'].isna().sum())}"
      f" of {len(sta)} (populated only for churners)")

print("\n  CLTV is kept out too: it is another IBM model's output, not an observed "
      "\n  fact. Low AUC alone, but unknown provenance -> quarantine, don't gamble.")

=== leakage severity: AUC of each status field ALONE ===
   (a fully tuned model on real features reaches ~0.86)

  LEAK     Satisfaction Score   AUC 0.9504
  LEAK     Churn Score          AUC 0.9428
  suspect  CLTV                 AUC 0.5808

  LEAK     Customer Status      values ['Churned', 'Stayed', 'Joined']  <- restates the target
  LEAK     Churn Label          identical to Churn Value: True
  LEAK     Churn Category       null for 5174 of 7043 (populated only for churners)
  LEAK     Churn Reason         null for 5174 of 7043 (populated only for churners)

  CLTV is kept out too: it is another IBM model's output, not an observed 
  fact. Low AUC alone, but unknown provenance -> quarantine, don't gamble.


## 2. Column decisions

`City` is dropped despite being legitimate: 1,129 categories would either explode the
one-hot matrix or need target encoding, which is its own leakage risk. Geography is
kept in numeric form instead — `Latitude`, `Longitude`, and `Population` carry the
signal without the cardinality.

In [5]:
DEM_KEEP = ["Gender", "Age", "Married", "Number of Dependents"]
SVC_KEEP = [
    "Number of Referrals", "Tenure in Months", "Offer",
    "Phone Service", "Avg Monthly Long Distance Charges", "Multiple Lines",
    "Internet Type", "Avg Monthly GB Download",
    "Online Security", "Online Backup", "Device Protection Plan",
    "Premium Tech Support", "Streaming TV", "Streaming Movies", "Streaming Music",
    "Unlimited Data", "Contract", "Paperless Billing", "Payment Method",
    "Monthly Charge", "Total Charges", "Total Refunds",
    "Total Extra Data Charges", "Total Long Distance Charges",
]
LOC_KEEP = ["Latitude", "Longitude"]          # + Population, joined via Zip Code

LEAKY = ["Satisfaction Score", "Customer Status", "Churn Label", "Churn Score",
         "CLTV", "Churn Category", "Churn Reason"]

DROP_REASONS = {
    "Count":              "constant (= 1)",
    "Quarter":            "constant (Q3)",
    "Country":            "constant",
    "State":              "constant",
    "Location ID":        "identifier",
    "Service ID":         "identifier",
    "Status ID":          "identifier",
    "ID":                 "identifier",
    "Lat Long":           "string restatement of Latitude/Longitude",
    "City":               "1,129 categories - geography kept as Lat/Long/Population",
    "Zip Code":           "join key for Population; high-cardinality as a feature",
    "Under 30":           "exact duplicate of (Age < 30)",
    "Senior Citizen":     "exact duplicate of (Age >= 65)",
    "Dependents":         "exact duplicate of (Number of Dependents > 0)",
    "Referred a Friend":  "exact duplicate of (Number of Referrals > 0)",
    "Internet Service":   "exact duplicate of (Internet Type not null)",
    "Total Revenue":      "exact linear combination of four kept columns",
    "Satisfaction Score": "LEAKAGE - AUC 0.95 alone; recorded with the outcome",
    "Churn Score":        "LEAKAGE - AUC 0.94 alone; another model's output",
    "Customer Status":    "LEAKAGE - restates the target",
    "Churn Label":        "LEAKAGE - identical to Churn Value",
    "Churn Category":     "LEAKAGE - populated only for churners",
    "Churn Reason":       "LEAKAGE - populated only for churners",
    "CLTV":               "quarantined - another model's output, unknown provenance",
}

# Fail loudly if a column we planned to drop as a duplicate turned out to be distinct.
for col, verified in dup.items():
    assert verified, f"{col} was expected to be a duplicate but is NOT - re-check before dropping"
print("all duplicate assumptions verified against the data\n")

for c, why in DROP_REASONS.items():
    print(f"  drop  {c:22s} {why}")

all duplicate assumptions verified against the data

  drop  Count                  constant (= 1)
  drop  Quarter                constant (Q3)
  drop  Country                constant
  drop  State                  constant
  drop  Location ID            identifier
  drop  Service ID             identifier
  drop  Status ID              identifier
  drop  ID                     identifier
  drop  Lat Long               string restatement of Latitude/Longitude
  drop  City                   1,129 categories - geography kept as Lat/Long/Population
  drop  Zip Code               join key for Population; high-cardinality as a feature
  drop  Under 30               exact duplicate of (Age < 30)
  drop  Senior Citizen         exact duplicate of (Age >= 65)
  drop  Dependents             exact duplicate of (Number of Dependents > 0)
  drop  Referred a Friend      exact duplicate of (Number of Referrals > 0)
  drop  Internet Service       exact duplicate of (Internet Type not null)
  drop  Tot

## 3. Merge

In [6]:
# Population joins through Zip Code, which is then dropped.
loc_pop = loc.merge(pop[["Zip Code", "Population"]], on="Zip Code", how="left", validate="m:1")
assert loc_pop["Population"].notna().all(), "some zips have no population row"

model = (svc[[KEY] + SVC_KEEP]
         .merge(dem[[KEY] + DEM_KEEP], on=KEY, how="inner", validate="1:1")
         .merge(loc_pop[[KEY] + LOC_KEEP + ["Population"]], on=KEY, how="inner", validate="1:1")
         .merge(sta[[KEY, TARGET]], on=KEY, how="inner", validate="1:1"))

assert len(model) == len(svc) == 7043, f"join lost rows: {len(model)}"
assert model[KEY].is_unique, "duplicate customers after merge"

# Meaningful nulls -> explicit categories. Both mean "does not have it", not "unknown".
# NOT the string "None": pandas' default NA list parses that back as NaN on read_csv,
# so the file would round-trip into 5,403 nulls despite looking clean in memory.
model["Offer"] = model["Offer"].fillna("No Offer")
model["Internet Type"] = model["Internet Type"].fillna("No Internet")

print("merged:", model.shape)
print("nulls remaining:", int(model.isna().sum().sum()))
print("churn rate:", round(model[TARGET].mean(), 4))

merged: (7043, 33)
nulls remaining: 0
churn rate: 0.2654


In [7]:
# Guard: no leaky column can reach the modelling file, whatever anyone edits above.
leaked = [c for c in model.columns if c in LEAKY]
assert not leaked, f"LEAKAGE reached the model frame: {leaked}"

# Guard: nothing correlates with the target the way a leak would.
num = model.select_dtypes(include="number").drop(columns=[TARGET])
sus = {c: max(roc_auc_score(model[TARGET], num[c]), 1 - roc_auc_score(model[TARGET], num[c]))
       for c in num.columns}
worst = sorted(sus.items(), key=lambda kv: -kv[1])[:5]
print("highest single-feature AUC among kept numeric columns:")
for c, a in worst:
    print(f"  {c:34s} {a:.4f}" + ("   <-- INVESTIGATE" if a > 0.90 else ""))
assert worst[0][1] <= 0.90, f"suspicious single-feature AUC: {worst[0]}"
print("\nno kept feature behaves like a leak")

highest single-feature AUC among kept numeric columns:
  Tenure in Months                   0.7409
  Total Long Distance Charges        0.6572
  Total Charges                      0.6512
  Number of Referrals                0.6392
  Monthly Charge                     0.6208

no kept feature behaves like a leak


## 4. Write both files

The reference file keeps the leaky fields — quarantined, keyed by customer, and clearly
labelled. It exists so `Churn Reason` stays available for validating what a model
claims, which is genuinely useful and genuinely not a feature.

In [8]:
reference = sta[[KEY] + [c for c in LEAKY if c in sta.columns]].copy()

DATA.mkdir(exist_ok=True)
model.to_csv(OUT_MODEL, index=False)
reference.to_csv(OUT_REF, index=False)

# Verify by reading back what was actually written, not what we think we wrote.
rt = pd.read_csv(OUT_MODEL)
assert rt.shape == model.shape, f"round-trip mismatch: {rt.shape} vs {model.shape}"
assert not [c for c in rt.columns if c in LEAKY], "leak found in the file on disk"
# Check nulls on the RE-READ frame: fill tokens that collide with pandas' NA list
# (e.g. "None", "NA", "N/A") look fine in memory and reappear as NaN on read.
rt_nulls = rt.isna().sum()
assert rt_nulls.sum() == 0, f"nulls after round-trip:\n{rt_nulls[rt_nulls > 0]}"

print(f"{OUT_MODEL}  ->  {rt.shape[0]} rows x {rt.shape[1]} cols")
print(f"{OUT_REF}    ->  {reference.shape[0]} rows x {reference.shape[1]} cols  (QUARANTINED)")
print(f"\nfeatures: {rt.shape[1] - 2}  (excludes {KEY} and {TARGET})")
print(f"churn rate: {rt[TARGET].mean():.4f}")
print("\ncolumns written to the model file:")
for c in rt.columns:
    kind = "TARGET" if c == TARGET else ("key" if c == KEY else str(rt[c].dtype))
    print(f"  {c:36s} {kind}")

data/telco_churn_model.csv  ->  7043 rows x 33 cols
data/telco_churn_reference.csv    ->  7043 rows x 8 cols  (QUARANTINED)

features: 31  (excludes Customer ID and Churn Value)
churn rate: 0.2654

columns written to the model file:
  Customer ID                          key
  Number of Referrals                  int64
  Tenure in Months                     int64
  Offer                                str
  Phone Service                        str
  Avg Monthly Long Distance Charges    float64
  Multiple Lines                       str
  Internet Type                        str
  Avg Monthly GB Download              int64
  Online Security                      str
  Online Backup                        str
  Device Protection Plan               str
  Premium Tech Support                 str
  Streaming TV                         str
  Streaming Movies                     str
  Streaming Music                      str
  Unlimited Data                       str
  Contract                

## 5. What changed vs the old single-table file

The old `Telco_customer_churn.xlsx` had 33 columns and no usage data. Everything listed
as *new* below is information the previous model could not see at all — which matters,
because every attempt to squeeze more out of the old features (cluster labels, UMAP
coordinates, per-segment thresholds) failed for exactly one reason: it was all
re-arrangements of data the model already had. These columns are not.

In [9]:
old = pd.read_excel("Telco_customer_churn.xlsx")
old_cols = set(old.columns)

new_feats = [c for c in model.columns
             if c not in old_cols and c not in (KEY, TARGET)]
print(f"{len(new_feats)} features the old file did not have:\n")
for c in new_feats:
    print(f"  {c:36s} {str(model[c].dtype):>8}   e.g. {model[c].dropna().iloc[0]!r}")

print("\nThe ones to watch:")
print("  Avg Monthly GB Download   - real usage. #2 stated churn reason is"
      " 'Competitor offered\n                              higher download speeds';"
      " a heavy downloader on DSL is\n                              exactly who a fibre"
      " rival poaches. Previously invisible.")
print("  Offer                     - which retention treatment each customer got.")
print("  Number of Referrals       - advocacy signal.")
print("  Total Refunds / Extra Data Charges - service-failure and bill-shock proxies.")
print("\nStill missing: support-interaction data. The #1 stated reason is 'Attitude of"
      "\nsupport person' (13% of churners) and nothing in these five tables measures it.")

18 features the old file did not have:

  Number of Referrals                     int64   e.g. np.int64(0)
  Tenure in Months                        int64   e.g. np.int64(1)
  Offer                                     str   e.g. 'No Offer'
  Avg Monthly Long Distance Charges     float64   e.g. np.float64(0.0)
  Internet Type                             str   e.g. 'DSL'
  Avg Monthly GB Download                 int64   e.g. np.int64(8)
  Device Protection Plan                    str   e.g. 'Yes'
  Premium Tech Support                      str   e.g. 'No'
  Streaming Music                           str   e.g. 'No'
  Unlimited Data                            str   e.g. 'No'
  Monthly Charge                        float64   e.g. np.float64(39.65)
  Total Refunds                         float64   e.g. np.float64(0.0)
  Total Extra Data Charges                int64   e.g. np.int64(20)
  Total Long Distance Charges           float64   e.g. np.float64(0.0)
  Age                                

## 6. Correlation matrix — a redundancy audit of the final column set

Section 1 proved the **exact** duplicates. This is the follow-up question: did anything
*nearly* duplicated survive? Two columns at r = 0.98 are not caught by an equality check,
but they split a feature's importance across two rows just the same.

**How the mixed types are handled.** A raw `.corr()` here would silently return a 16×16
matrix of the numeric columns and quietly ignore 17 others — pandas 3.0 gives the text
columns a `str` dtype, and `numeric_only=True` drops them without a word. So each column
is encoded to the scale its type actually supports:

| Type | Encoding | What Pearson means on it |
|---|---|---|
| numeric (15) | as-is | ordinary correlation |
| binary Yes/No (13) | 0/1 | the **phi coefficient** — Pearson on two binaries is exactly this |
| `Contract` | ordinal code 0/1/2 | valid: churn falls monotonically (45.8% → 10.7% → 2.5%) |
| `Offer`, `Internet Type`, `Payment Method` | **excluded** | no ordering exists, so a correlation would be an artefact of label order |

The three excluded columns are not ignored — they get Cramér's V below, which is the
right measure for them.

**What this matrix cannot do.** Correlation is *pairwise* and *linear*. `Total Revenue`
is an exact function of four other columns, yet it correlates perfectly with none of them
individually. That is precisely why the explicit checks in section 1 exist. This matrix
supplements them; it does not replace them.

In [10]:
import plotly.graph_objects as go

# --- dataviz palette: reference instance, light surface ---
SURFACE, INK, SECONDARY, MUTED = "#fcfcfb", "#0b0b0b", "#52514e", "#898781"
GRID, BASELINE = "#e1e0d9", "#c3c2b7"

corr_df = model.drop(columns=[KEY]).copy()

# Encode each column on the scale its type supports. Do NOT lean on numeric_only=True:
# in pandas 3.0 the text columns carry a 'str' dtype and would be dropped in silence.
non_numeric = [c for c in corr_df.columns if not pd.api.types.is_numeric_dtype(corr_df[c])]
binary_cols = [c for c in non_numeric if corr_df[c].nunique() == 2]
nominal_cols = [c for c in non_numeric if corr_df[c].nunique() > 2 and c != "Contract"]

for c in binary_cols:
    levels = sorted(corr_df[c].unique())
    positive = "Yes" if "Yes" in levels else levels[-1]
    corr_df[c] = (corr_df[c] == positive).astype(int)

# Contract is ordinal: churn falls monotonically across the levels, so an integer code
# carries real information. The nominal columns have no such order and are excluded.
corr_df["Contract"] = corr_df["Contract"].map({"Month-to-Month": 0, "One Year": 1, "Two Year": 2})
corr_df = corr_df.drop(columns=nominal_cols)

assert corr_df.notna().all().all(), "encoding produced NaNs"
assert all(pd.api.types.is_numeric_dtype(corr_df[c]) for c in corr_df.columns), \
    "a non-numeric column survived encoding — the matrix would silently drop it"

corr = corr_df.corr()
print(f"{corr.shape[0]} columns in the matrix "
      f"({len(binary_cols)} binary as phi, 1 ordinal, {corr.shape[0] - len(binary_cols) - 1} numeric)")
print(f"excluded (no ordering to correlate): {nominal_cols}")

29 columns in the matrix (13 binary as phi, 1 ordinal, 15 numeric)
excluded (no ordering to correlate): ['Offer', 'Internet Type', 'Payment Method']


In [11]:
# The audit itself: what is most nearly redundant?
off = corr.where(~np.eye(len(corr), dtype=bool))
ranked, seen = [], set()
for (a, b), v in off.abs().stack().sort_values(ascending=False).items():
    if frozenset((a, b)) in seen:
        continue
    seen.add(frozenset((a, b)))
    ranked.append({"feature A": a, "feature B": b, "r": corr.loc[a, b]})

top = pd.DataFrame(ranked).head(12)
print("=== most correlated pairs (the near-duplicate audit) ===\n")
print(top.to_string(index=False, formatters={"r": "{:+.3f}".format}))

NEAR_DUP = 0.95
worst = top.iloc[0]
assert abs(worst["r"]) < NEAR_DUP, (
    f"near-duplicate survived the section-1 audit: {worst['feature A']} <-> "
    f"{worst['feature B']} at r={worst['r']:+.3f}")
print(f"\nPASS: no pair reaches |r| >= {NEAR_DUP}. The exact-duplicate audit in section 1")
print(f"      did not miss a near-twin. Strongest is {abs(worst['r']):.3f}.")

=== most correlated pairs (the near-duplicate audit) ===

                        feature A                   feature B      r
                         Latitude                   Longitude -0.886
                 Streaming Movies             Streaming Music +0.849
                 Tenure in Months               Total Charges +0.826
      Total Long Distance Charges            Tenure in Months +0.674
              Number of Referrals                     Married +0.673
                    Total Charges              Monthly Charge +0.651
                         Contract            Tenure in Months +0.647
                     Streaming TV              Monthly Charge +0.630
                 Streaming Movies              Monthly Charge +0.627
      Total Long Distance Charges               Total Charges +0.610
Avg Monthly Long Distance Charges Total Long Distance Charges +0.600
                   Monthly Charge              Unlimited Data +0.582

PASS: no pair reaches |r| >= 0.95. The exact

In [12]:
# Grouped so blocks of related columns sit together — an alphabetical axis would
# scatter the charge family across the matrix and hide exactly what we are looking for.
ORDER = [
    "Tenure in Months", "Contract",
    "Monthly Charge", "Total Charges", "Avg Monthly Long Distance Charges",
    "Total Long Distance Charges", "Total Refunds", "Total Extra Data Charges",
    "Avg Monthly GB Download", "Unlimited Data",
    "Phone Service", "Multiple Lines", "Online Security", "Online Backup",
    "Device Protection Plan", "Premium Tech Support",
    "Streaming TV", "Streaming Movies", "Streaming Music",
    "Paperless Billing", "Number of Referrals",
    "Age", "Married", "Number of Dependents", "Gender",
    "Latitude", "Longitude", "Population",
    TARGET,
]
assert set(ORDER) == set(corr.columns), f"ORDER mismatch: {set(corr.columns) ^ set(ORDER)}"

C = corr.loc[ORDER, ORDER].copy()
z = C.values.astype(float)
z[np.triu_indices_from(z, k=0)] = np.nan   # upper half mirrors the lower; the diagonal is always 1

# Diverging ramp: blue <-> red poles, neutral gray midpoint. Polarity is the job here --
# a sequential ramp would make -0.9 and +0.9 look like opposite ends of one magnitude
# instead of opposite *signs*. Both arms verified monotone in lightness.
DIVERGING = [
    [0.000, "#0d366b"], [0.125, "#1c5cab"], [0.250, "#2a78d6"], [0.375, "#9ec5f4"],
    [0.500, "#f0efec"],
    [0.625, "#f7bcbb"], [0.750, "#e34948"], [0.875, "#b02a2a"], [1.000, "#7d1a1a"],
]

fig = go.Figure(go.Heatmap(
    z=z, x=ORDER, y=ORDER,
    colorscale=DIVERGING, zmid=0, zmin=-1, zmax=1,
    xgap=2, ygap=2,                       # 2px surface gap between fills
    hovertemplate="<b>%{y}</b><br>vs %{x}<br>r = %{z:+.3f}<extra></extra>",
    colorbar=dict(title=dict(text="Pearson r", font=dict(color=MUTED, size=11)),
                  tickfont=dict(color=MUTED, size=10), outlinewidth=0,
                  thickness=14, len=0.75, tickvals=[-1, -0.5, 0, 0.5, 1]),
))

fig.update_layout(
    title=dict(text="Correlation matrix — final modelling columns "
                    "(binary as phi, Contract ordinal-coded)",
               font=dict(color=INK, size=16)),
    plot_bgcolor=SURFACE, paper_bgcolor=SURFACE,
    font=dict(family='system-ui, -apple-system, "Segoe UI", sans-serif'),
    height=720, margin=dict(t=70, r=20, b=175, l=215),
    xaxis=dict(tickangle=-45, tickfont=dict(color=SECONDARY, size=10),
               showgrid=False, linecolor=BASELINE),
    # No scaleanchor: forcing square cells strands ~30% of the width as dead space.
    # Cell shape carries no meaning here — the color does.
    yaxis=dict(autorange="reversed", tickfont=dict(color=SECONDARY, size=10),
               showgrid=False, linecolor=BASELINE),
)
fig.show()

In [13]:
# The three columns Pearson cannot take. Cramer's V: 0 = independent, 1 = one column
# determines the other. Same question as the matrix, correct measure for nominal data.
def cramers_v(a, b):
    ct = pd.crosstab(a, b).values
    n = ct.sum()
    expected = np.outer(ct.sum(1), ct.sum(0)) / n
    stat = ((ct - expected) ** 2 / expected).sum()
    r, k = ct.shape
    return float(np.sqrt((stat / n) / max(min(r - 1, k - 1), 1)))


cat_cols = nominal_cols + ["Contract", TARGET]
V = pd.DataFrame(
    [[cramers_v(model[a], model[b]) for b in cat_cols] for a in cat_cols],
    index=cat_cols, columns=cat_cols)

print("=== Cramer's V — the columns Pearson cannot take ===\n")
print(V.round(3).to_string())

pairs = [(a, b, V.loc[a, b]) for i, a in enumerate(cat_cols) for b in cat_cols[i + 1:]]
worst_v = max(pairs, key=lambda t: t[2])
assert worst_v[2] < NEAR_DUP, f"nominal near-duplicate: {worst_v}"
print(f"\nPASS: strongest nominal association is {worst_v[0]} <-> {worst_v[1]} "
      f"at V = {worst_v[2]:.3f}")
print("      — related, not redundant. Nothing to drop.")

=== Cramer's V — the columns Pearson cannot take ===

                Offer  Internet Type  Payment Method  Contract  Churn Value
Offer           1.000          0.038           0.073     0.340        0.262
Internet Type   0.038          1.000           0.235     0.166        0.305
Payment Method  0.073          0.235           1.000     0.118        0.219
Contract        0.340          0.166           0.118     1.000        0.453
Churn Value     0.262          0.305           0.219     0.453        1.000

PASS: strongest nominal association is Contract <-> Churn Value at V = 0.453
      — related, not redundant. Nothing to drop.


### What the matrix says

**Nothing is redundant.** The strongest pair reaches |r| = 0.886, well under the 0.95
guard, so the exact-duplicate audit in section 1 did not miss a near-twin. The column set
is clean.

The three strongest pairs are each worth understanding, because none of them is a reason
to drop a column:

- **`Latitude` ↔ `Longitude`, r = −0.886.** Startling until you remember every customer is
  in California, which lies on a diagonal. This is the *shape of the state*, not
  redundancy — the two columns still pin down different places. Drop one and you lose
  location entirely.
- **`Streaming Movies` ↔ `Streaming Music`, r = +0.849.** People who stream, stream. Two
  separate subscriptions that happen to co-occur; either can be held without the other.
- **`Total Charges` ↔ `Tenure in Months`, r = +0.826.** Arithmetic: a bill accumulates over
  time. Structural, and the residual — *charges high for how long they have been here* —
  is exactly the spend-intensity signal `avg_monthly_spend` extracts in the modelling
  notebook.

**On the target column.** `Contract` at −0.435 is the strongest linear relationship with
churn, and that is a healthy number. The leak check in section 3 is the one that matters
for safety — a column correlating 0.95 with churn would have been caught there by AUC,
which does not assume linearity. Read this matrix for redundancy between features; read
the AUC sweep for leakage.

**A caveat worth keeping.** Low correlation with churn is not evidence a feature is
useless. `Avg Monthly GB Download` correlates just +0.049 with churn, yet it is one of the
most promising new columns — heavy downloaders churn only *in combination with* the wrong
internet type, and a pairwise linear coefficient cannot see an interaction. Correlation
audits redundancy. It does not rank features.

---
### Using these files

```python
df = pd.read_csv("data/telco_churn_model.csv")
y  = df["Churn Value"]
X  = df.drop(columns=["Churn Value", "Customer ID"])
```

Feature engineering (tenure buckets, spend ratios, service counts) belongs in the
modelling notebook, not here — this file stays a clean source of truth.

`telco_churn_reference.csv` is for validating results after the fact. If you ever find
yourself merging it into `X`, stop: that is how a 0.95-AUC model that fails in
production gets built.